<a href="https://colab.research.google.com/github/roopaam/MB-Proxy/blob/main/Adult_Stability_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="pgmpy")
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.preprocessing import KBinsDiscretizer, LabelEncoder

from pgmpy.estimators import PC, HillClimbSearch, BDeu
from pgmpy.models import DiscreteBayesianNetwork

# -----------------------------
# 1) Load UCI Adult Dataset
# -----------------------------

def load_adult_data() -> pd.DataFrame:
    """
    Loads the UCI Adult dataset from the official repository.
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    columns = [
        "age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
        "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
        "hours-per-week", "native-country", "income"
    ]

    # Load and strip whitespace
    df = pd.read_csv(url, names=columns, sep=",", skipinitialspace=True, na_values="?")

    # Simple cleaning: drop rows with missing values to maintain causal structure integrity
    df = df.dropna()

    # Rename target for consistency
    df["income"] = df["income"].apply(lambda x: 1 if ">50K" in x else 0)

    # Remove 'fnlwgt' as it is a sampling weight, not a causal feature
    if "fnlwgt" in df.columns:
        df = df.drop(columns=["fnlwgt"])

    return df

# -----------------------------
# 2) Preprocess to discrete ints (Generic)
# -----------------------------

@dataclass
class PreprocessConfig:
    n_bins: int = 5
    bin_strategy: str = "quantile"
    max_rows: Optional[int] = None

def preprocess_discrete_for_pgmpy(df: pd.DataFrame, target_col: str, cfg: PreprocessConfig) -> Tuple[pd.DataFrame, List[str]]:
    out = df.copy()

    if cfg.max_rows is not None and len(out) > cfg.max_rows:
        out = out.sample(n=cfg.max_rows, random_state=42).reset_index(drop=True)

    # Identify numeric columns excluding target
    numeric_cols = [c for c in out.columns if c != target_col and pd.api.types.is_numeric_dtype(out[c])]
    cat_cols = [c for c in out.columns if c not in numeric_cols]

    # Discretize numeric columns
    if numeric_cols:
        binner = KBinsDiscretizer(
            n_bins=cfg.n_bins,
            encode="ordinal",
            strategy=cfg.bin_strategy,
        )
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            out[numeric_cols] = binner.fit_transform(out[numeric_cols])
        out[numeric_cols] = out[numeric_cols].astype(int)

    # Label encode categorical columns
    for c in cat_cols:
        le = LabelEncoder()
        out[c] = le.fit_transform(out[c].astype(str))

    feature_cols = [c for c in out.columns if c != target_col]
    return out, feature_cols

# -----------------------------
# 3) Graph learning helpers
# -----------------------------

def pc_stable_markov_blanket(data: pd.DataFrame, target_col: str, alpha: float, max_cond_vars: int = 2) -> List[str]:
    est = PC(data)
    model = est.estimate(
        variant="stable",
        ci_test="chi_square",
        significance_level=alpha,
        max_cond_vars=max_cond_vars,
        show_progress=False,
    )
    try:
        dag = model.to_dag()
        edges = list(dag.edges())
    except Exception:
        edges = list(model.edges())

    bn = DiscreteBayesianNetwork(edges)
    mb = bn.get_markov_blanket(target_col)
    return sorted(set(mb))

def hc_bdeu_markov_blanket(data: pd.DataFrame, target_col: str, ess: int = 10, max_indegree: int = 2) -> List[str]:
    scoring = BDeu(data, equivalent_sample_size=ess)
    est = HillClimbSearch(data)
    model = est.estimate(scoring_method=scoring, max_indegree=max_indegree, show_progress=False)
    edges = list(model.edges())
    bn = DiscreteBayesianNetwork(edges)
    mb = bn.get_markov_blanket(target_col)
    return sorted(set(mb))

# -----------------------------
# 4) Bootstrap stability
# -----------------------------

def stability_analysis_pc_hc(
    data: pd.DataFrame,
    target_col: str,
    feature_cols: List[str],
    pc_alpha: float,
    hc_max_indegree: int,
    n_bootstraps: int = 50,
    sample_frac: float = 0.30,
    random_state: int = 42,
    pc_max_cond_vars: int = 2,
    hc_ess: int = 10,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rng = np.random.default_rng(random_state)
    n = len(data)
    rows = []

    for b in tqdm(range(1, n_bootstraps + 1), desc="Bootstraps"):
        idx = rng.choice(n, size=int(sample_frac * n), replace=True)
        sample = data.iloc[idx].reset_index(drop=True)

        # PC-Stable
        try:
            mb_pc = pc_stable_markov_blanket(sample, target_col, alpha=pc_alpha, max_cond_vars=pc_max_cond_vars)
        except Exception:
            mb_pc = []

        for f in feature_cols:
            rows.append({"bootstrap": b, "algorithm": "PC-Stable", "feature": f, "selected": int(f in mb_pc)})

        # HC-BDeu
        try:
            mb_hc = hc_bdeu_markov_blanket(sample, target_col, ess=hc_ess, max_indegree=hc_max_indegree)
        except Exception:
            mb_hc = []

        for f in feature_cols:
            rows.append({"bootstrap": b, "algorithm": "HC-BDeu", "feature": f, "selected": int(f in mb_hc)})

    raw_df = pd.DataFrame(rows)
    summary_df = (
        raw_df.groupby(["algorithm", "feature"], as_index=False)
        .agg(selection_frequency=("selected", "mean"), variance=("selected", "var"), n=("selected", "count"))
        .sort_values(["algorithm", "selection_frequency"], ascending=[True, False])
        .reset_index(drop=True)
    )
    return summary_df, raw_df

# -----------------------------
# 5) Run Sweep
# -----------------------------

if __name__ == "__main__":
    # (A) Load Adult Data
    df = load_adult_data()
    target_col = "income"

    # (B) Preprocess
    cfg = PreprocessConfig(n_bins=5, bin_strategy="quantile", max_rows=30000)
    disc_df, feature_cols = preprocess_discrete_for_pgmpy(df, target_col, cfg)

    print("Adult Dataset loaded:", disc_df.shape)
    print("Features:", feature_cols)

    # Define sweep ranges
    pc_alphas = [0.01, 0.05, 0.10]
    hc_indegrees = [2, 3, 4]

    all_summaries = []
    all_raw_data = []

    print("\nStarting Stability Sweep for Adult Dataset...")

    for alpha in pc_alphas:
        for indegree in hc_indegrees:
            print(f"\n>>> Testing: PC Alpha={alpha}, HC Max Indegree={indegree}")

            summary, raw = stability_analysis_pc_hc(
                data=disc_df,
                target_col=target_col,
                feature_cols=feature_cols,
                n_bootstraps=50,
                sample_frac=0.30,
                random_state=42,
                pc_alpha=alpha,
                pc_max_cond_vars=2,
                hc_ess=10,
                hc_max_indegree=indegree,
            )

            summary['pc_alpha'] = alpha
            summary['hc_max_indegree'] = indegree
            raw['pc_alpha'] = alpha
            raw['hc_max_indegree'] = indegree

            all_summaries.append(summary)
            all_raw_data.append(raw)

            # Quick Print
            for algo in ["PC-Stable", "HC-BDeu"]:
                print(f"[{algo}] Top features:")
                print(summary[summary["algorithm"] == algo].head(5)[["feature", "selection_frequency"]])

    # Consolidate and Save
    final_summary = pd.concat(all_summaries, ignore_index=True)
    final_raw = pd.concat(all_raw_data, ignore_index=True)

    final_summary.to_csv("adult_mb_stability_sweep_summary.csv", index=False)
    final_raw.to_csv("adult_mb_stability_sweep_raw.csv", index=False)

    print("\n" + "="*40)
    print("Sweep Complete. Files saved: adult_mb_stability_sweep_summary.csv")

Adult Dataset loaded: (30000, 14)
Features: ['age', 'workclass', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']
>>> Starting PC-Stable Sensitivity Sweep (Alpha)...
Running PC-Stable with Alpha = 0.01


Bootstraps:   0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   2%|▏         | 1/50 [02:57<2:24:35, 177.04s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   4%|▍         | 2/50 [05:52<2:20:45, 175.94s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is depr

Running PC-Stable with Alpha = 0.05


Bootstraps:   0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   2%|▏         | 1/50 [03:02<2:28:41, 182.07s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   4%|▍         | 2/50 [06:14<2:30:28, 188.09s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is depr

Running PC-Stable with Alpha = 0.1


...

Bootstraps:   0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   2%|▏         | 1/50 [03:02<2:29:22, 182.90s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   4%|▍         | 2/50 [06:16<2:31:14, 189.06s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is depr

Running HC-BDeu with Max In-Degree = 3


Bootstraps:   0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   2%|▏         | 1/50 [03:07<2:32:47, 187.09s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   4%|▍         | 2/50 [06:23<2:33:53, 192.37s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is depr

Running HC-BDeu with Max In-Degree = 4


Bootstraps:   0%|          | 0/50 [00:00<?, ?it/s]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   2%|▏         | 1/50 [03:04<2:30:30, 184.29s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is deprecated and will be removed in v1.3.0. Please use pgmpy.causal_discovery.PC instead.
  est = PC(data)
/tmp/ipykernel_11139/3353118918.py:110: FutureWarning: HillClimbSearch is deprecated and will be removed in v1.3.0. Please use
            pgmpy.causal_discovery.HillClimbSearch instead.
  est = HillClimbSearch(data)
Bootstraps:   4%|▍         | 2/50 [06:18<2:32:03, 190.08s/it]/tmp/ipykernel_11139/3353118918.py:90: FutureWarning: PC is depr